# Install detectron2

In [ ]:
!python -m pip install pyyaml==5.1
# Detectron2 has not released pre-built binaries for the latest pytorch (https://github.com/facebookresearch/detectron2/issues/4053)
# so we install from source instead. This takes a few minutes.
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

# Install pre-built detectron2 that matches pytorch version, if released:
# See https://detectron2.readthedocs.io/tutorials/install.html for instructions
#!pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/{CUDA_VERSION}/{TORCH_VERSION}/index.html

# exit(0)  # After installation, you may need to "restart runtime" in Colab. This line can also restart runtime

In [ ]:
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

In [ ]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog
from torch.nn import functional as F

# Run a pre-trained detectron2 model

In [ ]:
import glob
from IPython.display import Image, display
import random

We first download an image from the COCO dataset:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# %cd /content/detectron2/sparseinst

In [ ]:
path="/content/drive/MyDrive/VD Synopsis/patches"
images = [f for f in os.listdir(path) if os.path.splitext(f)[-1] == '.png']
for i in images:
  im=cv2.imread('/content/drive/MyDrive/VD Synopsis/patches/'+str(i))
  tmp = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
  _,alpha = cv2.threshold(tmp,0,255,cv2.THRESH_BINARY)
  b, g, r = cv2.split(im)
  rgba = [b,g,r, alpha]
  dst = cv2.merge(rgba,4)
  cv2.imwrite('/content/drive/MyDrive/VD Synopsis/patches_png/'+str(i)[0:-4]+'.png',dst)

In [ ]:
# !python tools/test_net.py --config-file <CONFIG> MODEL.WEIGHTS <MODEL-PATH> INPUT.MIN_SIZE_TEST 512
# !python tools/test_net.py --config-file /content/sparseinst/configs/sparse_inst_r50_giam.yaml MODEL.WEIGHTS /content/drive/MyDrive/sparse_inst_r50_giam_ceaffc.pth INPUT.MIN_SIZE_TEST 512

In [ ]:
# !wget http://images.cocodataset.org/val2017/000000439715.jpg -q -O input.jpg

path="/content/drive/MyDrive/VD Synopsis/patches"
images = [f for f in os.listdir(path) if os.path.splitext(f)[-1] == '.png']

for i  in images:

  im=cv2.imread('/content/drive/MyDrive/VD Synopsis/patches/'+str(i))
  cfg = get_cfg()
  # add project-specific config (e.g., TensorMask) here if you're not running a model in detectron2's core library
  cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_1x.yaml"))
  cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # set threshold for this model
  # Find a model from detectron2's model zoo. You can use the https://dl.fbaipublicfiles... url as well
  cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_1x.yaml")
  predictor = DefaultPredictor(cfg)
  outputs = predictor(im)

  # We can use `Visualizer` to draw the predictions on the image.
  # v = Visualizer(im[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
  # out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
  # cv2_imshow(out.get_image()[:, :, ::-1])

  from PIL import Image
  #Get the mask
  masks=np.asarray(outputs["instances"].pred_masks.to("cpu"))
  #Pick an item to the mask

  try:
    item_mask=masks[0]
  except:
    continue

  # Get the true bouding box of the mask
  segmentation=np.where(item_mask -- True)
  x_min=int(np.min(segmentation[1]))
  x_max=int(np.max(segmentation[1]))
  y_min=int(np.min(segmentation[0]))
  y_max=int(np.max(segmentation[0]))

  #Create croped image from the just the portion of the image 
  cropped=Image.fromarray(im[y_min:y_max,x_min:x_max,:], mode='RGB')

  #Create a PIL image out of the mask
  mask = Image.fromarray((item_mask * 255).astype('uint8'))

  #Crop the mask to match the cropped image
  cropped_mask=mask.crop((x_min,y_min,x_max,y_max))

  #Load in a backgorund image and choose a paste position
  background=Image.fromarray((im*0).astype('uint8'))
  
  #Create a new foreground image as large as the composite and paste the cropped image on top 
  new_fg_image = Image.new('RGB', background.size)
  new_fg_image.paste(cropped)

  # Create a new alpha mask as large as the composite and paste the cropped mask
  new_alpha_mask = Image.new('RGB', background.size)
  new_alpha_mask.paste(cropped_mask)

  # Compose the foreground and background using the alpha mask
  composite = Image.composite(new_fg_image,background,mask)
  
  src=np.array(composite)
  tmp = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
  _, alpha = cv2.threshold(tmp, 0, 255, cv2.THRESH_BINARY)
  b, g, r = cv2.split(src)
  rgba = [b, g, r, alpha]
  dst = cv2.merge(rgba, 4)

  cv2.imwrite('/content/drive/MyDrive/VD Synopsis/Seg_patches/'+str(i[0:-4])+'.png', dst)

In [ ]:
img=cv2.imread('/content/png-transparent-chinese-beagle-chinese-crested-dog-puppy-maltese-dog-pet-sitting-bark-dog-walking.png')
1-img[...,-1]

In [ ]:
# look at the outputs. See https://detectron2.readthedocs.io/tutorials/models.html#model-output-format for specification
print(outputs["instances"].pred_classes)
print(outputs["instances"].pred_boxes)

In [ ]:
# We can use `Visualizer` to draw the predictions on the image.
v = Visualizer(im[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
cv2_imshow(out.get_image()[:, :, ::-1])

In [ ]:
from PIL import Image

#Get the mask
masks=np.asarray(outputs["instances"].pred_masks.to("cpu"))

#Pick an item to the mask
item_mask=masks[0]

#Get the true bouding box of the mask
segmentation=np.where(item_mask -- True)
x_min=int(np.min(segmentation[1]))
x_max=int(np.max(segmentation[1]))
y_min=int(np.min(segmentation[0]))
y_max=int(np.max(segmentation[0]))

#Create croped image from the just the portion of the image 
cropped=Image.fromarray(im[y_min:y_max,x_min:x_max,:], mode='RGB')

#Create a PIL image out of the mask
mask = Image.fromarray((item_mask * 255).astype('uint8'))

#Crop the mask to match the cropped image
cropped_mask=mask.crop((x_min,y_min,x_max,y_max)) 

#Load in a backgorund image and choose a paste position
background=Image.fromarray(im,mode='RGB')

# paste_position=(80,90)

#Create a new foreground image as large as the composite and paste the cropped image on top 
new_fg_image = Image.new('RGB', background.size)
new_fg_image.paste(cropped)

# Create a new alpha mask as large as the composite and paste the cropped mask
new_alpha_mask = Image.new('RGB', background.size)
new_alpha_mask.paste(cropped_mask)

# Compose the foreground and background using the alpha mask
composite = Image.composite(new_fg_image,new_alpha_mask,mask)

# Display the image
image=np.array(composite)
cv2_imshow(image)
cv2.imwrite('/content/drive/MyDrive/VD Synopsis/Segment_patches/1.jpg', image)

In [ ]:
# import cv2
# import numpy as np
# image = cv2.imread('/content/input.jpg')

# # Fill the black background with white color
# #cv2.floodFill(image, None, seedPoint=(0, 0), newVal=(0, 0, 255), loDiff=(2, 2, 2), upDiff=(2, 2, 2))  # Not working!

# hsv_img = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)  # rgb to hsv color space

# s_ch = hsv_img[:, :, 1]  # Get the saturation channel

# thesh = cv2.threshold(s_ch, 5, 255, cv2.THRESH_BINARY)[1]  # Apply threshold - pixels above 5 are going to be 255, other are zeros.
# thesh = cv2.morphologyEx(thesh, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))  # Apply opening morphological operation for removing artifacts.

# cv2.floodFill(thesh, None, seedPoint=(0, 0), newVal=128, loDiff=1, upDiff=1)  # Fill the background in thesh with the value 128 (pixel in the foreground stays 0.

# image[thesh == 128] = (0, 0, 255)  # Set all the pixels where thesh=128 to red.

# cv2.imwrite('tulips_red_bg.jpg', image)  # Save the output image.